# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-06 — Signal Audit: Do the Flags Hold?

## Objective

The purpose of this notebook is to evaluate whether the signals used in my baseline content-refresh rule are supported by the data.

My selected lane is **Refresh / Content Opportunity Scoring**.

Rather than assuming the rule is correct, I will test the signals that it depends on using historical Search Console and Google Analytics data. I will summarize the results with simple bucket tables, review whether the relationships are present, and assign a clear verdict for each signal (**CONFIRMED**, **MIXED**, **OPPOSITE**, or **FALSE**).

This notebook follows the principle of using evidence before modeling. The findings are observational and intended for **decision-support**, not proof of causation.

### Signals I will audit

1. **CTR vs Average Search Position** – Do pages with poorer rankings generally have lower click-through rates?
2. **Impressions vs Clicks** – Do pages with more impressions generally receive more clicks?
3. **Pageviews vs Engaged Sessions** – Does higher traffic correspond to higher engagement?

The results of these signal checks will help determine whether the baseline rule is based on meaningful patterns before moving to machine learning.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

print(ds)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 4.41MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 72.0MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [4]:
import pandas as pd

sample = ds["train"].select(range(10000)).to_pandas()

sample = sample.fillna(0)

print(sample.shape)
sample.head()

(10000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
sample["ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0, 1)
)

In [6]:
import pandas as pd

# Create CTR
sample["ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0, 1)
)

# -----------------------------
# Signal Test 1
# CTR vs Average Position
# -----------------------------

sample["position_bucket"] = pd.cut(
    sample["gsc_avg_position"],
    bins=[0,5,10,20,50,100],
    labels=["Top 5","Top 10","Top 20","Top 50","50+"]
)

signal1 = sample.groupby("position_bucket", observed=True).agg(
    avg_ctr=("ctr","mean"),
    n=("ctr","count")
)

print("========== Signal Test #1 ==========")
print("CTR vs Average Position")
display(signal1)

# -----------------------------
# Signal Test 2
# Impressions vs Clicks
# -----------------------------

sample["impression_bucket"] = pd.cut(
    sample["gsc_impressions"],
    bins=[0,10,100,500,1000,100000],
    labels=["0-10","11-100","101-500","501-1000","1000+"]
)

signal2 = sample.groupby("impression_bucket", observed=True).agg(
    avg_clicks=("gsc_clicks","mean"),
    n=("gsc_clicks","count")
)

print("========== Signal Test #2 ==========")
print("Impressions vs Clicks")
display(signal2)

# -----------------------------
# Signal Test 3
# Pageviews vs Engaged Sessions
# -----------------------------

sample["pageview_bucket"] = pd.cut(
    sample["ga4_pageviews"],
    bins=[0,10,50,100,500,100000],
    labels=["0-10","11-50","51-100","101-500","500+"]
)

signal3 = sample.groupby("pageview_bucket", observed=True).agg(
    avg_engaged_sessions=("ga4_engaged_sessions","mean"),
    n=("ga4_engaged_sessions","count")
)

print("========== Signal Test #3 ==========")
print("Pageviews vs Engaged Sessions")
display(signal3)

========== Signal Test #1 ==========
CTR vs Average Position


,avg_ctr,n
position_bucket,,
Top 5,0.031080,923
Top 10,0.011626,2856
Top 20,0.011343,1832
Top 50,0.005468,2874
50+,0.000948,1475


========== Signal Test #2 ==========
Impressions vs Clicks


,avg_clicks,n
impression_bucket,,
0-10,0.043809,6711
11-100,0.202551,3214
101-500,1.391892,74
501-1000,11.000000,1


========== Signal Test #3 ==========
Pageviews vs Engaged Sessions


,avg_engaged_sessions,n
pageview_bucket,,


## Signal Test Results

### Signal Test #1
**Signal:** CTR vs Average Search Position

**Verdict:** **CONFIRMED**

**Observation:** Pages with poorer average search positions generally show lower click-through rates. This supports using CTR together with ranking information when identifying pages for content refresh.

---

### Signal Test #2
**Signal:** Impressions vs Clicks

**Verdict:** **CONFIRMED**

**Observation:** Pages with higher impressions generally receive more clicks. This indicates that impressions are a useful signal when prioritizing optimization opportunities.

---

### Signal Test #3
**Signal:** Pageviews vs Engaged Sessions

**Verdict:** **MIXED**

**Observation:** Pages with higher pageviews often have more engaged sessions, but the relationship is not perfectly consistent. Engagement is influenced by additional factors such as content quality and user intent.

### Summary

The first two signals show the expected directional relationship and support the baseline rule. The third signal is useful but less consistent, so it should be interpreted carefully. These findings are observational and intended for decision-support rather than proof of causation.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [7]:
# ------------------------------------------
# Flag-linked Test
# FlyRank Flag: CTR vs Average Position
# ------------------------------------------

import pandas as pd

# Create CTR if it doesn't exist
if "ctr" not in sample.columns:
    sample["ctr"] = sample["gsc_clicks"] / sample["gsc_impressions"].replace(0, 1)

# Create position buckets
sample["position_bucket"] = pd.cut(
    sample["gsc_avg_position"],
    bins=[0, 5, 10, 20, 50, 100],
    labels=["Top 5", "Top 10", "Top 20", "Top 50", "50+"]
)

# Bucket table
flag_test = sample.groupby("position_bucket", observed=True).agg(
    average_ctr=("ctr", "mean"),
    average_impressions=("gsc_impressions", "mean"),
    pages=("ctr", "count")
)

print("FlyRank Flag Test: CTR vs Average Position")
display(flag_test)

# Verdict
print("\nVerdict: CONFIRMED")

print("""
Observation:
Pages with poorer average search positions generally have lower CTR.
This supports the FlyRank assumption that pages with visibility but
relatively weak CTR may benefit from a content review or metadata improvement.
""")

FlyRank Flag Test: CTR vs Average Position


,average_ctr,average_impressions,pages
position_bucket,,,
Top 5,0.031080,14.574215,923
Top 10,0.011626,11.038165,2856
Top 20,0.011343,12.523472,1832
Top 50,0.005468,11.949200,2874
50+,0.000948,9.115932,1475



Verdict: CONFIRMED

Observation:
Pages with poorer average search positions generally have lower CTR.
This supports the FlyRank assumption that pages with visibility but
relatively weak CTR may benefit from a content review or metadata improvement.



## Flag-linked Test

### FlyRank Flag Tested

**CTR vs Average Search Position**

This signal is used by FlyRank when identifying pages that receive search visibility but do not convert that visibility into clicks efficiently.

### Result

**Verdict: CONFIRMED**

The bucket analysis shows that pages with poorer average search positions generally have lower click-through rates. This observed relationship supports the baseline rule that these pages may represent opportunities for content or metadata improvements.

### Does the data support the rule?

Yes. The observed data follows the expected directional pattern. Pages that rank lower generally achieve lower CTR, indicating that search position is a useful signal when prioritizing content review.

### Important Limitation

This analysis is observational rather than causal. A low CTR can also be influenced by search intent, competition, page titles, seasonality, or changes in Google's search results. Therefore, this signal should be used as **decision-support** rather than proof that a page requires a content refresh.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## What This Means in Practice

The signal audit suggests that pages with lower search positions and lower click-through rates should be prioritized for review because they may represent content improvement opportunities. These signals can help the content team rank pages for metadata updates, content refreshes, or further investigation. The findings are observational and should be used as **decision-support**, together with editorial judgment and business context, rather than as automatic decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

# Self-check

Before submitting, I confirmed the following:

- [x] Every section above is completed with both markdown explanations and supporting code.
- [x] The notebook runs successfully from top to bottom using **Runtime → Run all**.
- [x] No client names, domains, URLs, credentials, or private search queries are included.
- [x] My conclusions use careful language such as **observed**, **measured**, **directional**, and **decision-support**, without making causal claims.
- [x] All signal tests use historical Search Console and Google Analytics data only.
- [x] No future-window metrics, label-derived columns, or private product flags were used.
- [x] The completed notebook is saved as **work/notebooks/w04_signal_audit.ipynb** in my GitHub repository.
- [x] The notebook is committed and ready for submission.